# Step 1 — Convert `ALLLOG/` to a LeRobot Dataset

This notebook converts data collected by `save_state_img.py` into the **LeRobot dataset format** that ACT, Diffusion Policy, and openpi all consume. Run this once per data-collection session; the training notebook will then load the result directly.

## Why this is a separate step

`save_state_img.py` writes a "don't lose anything" archive: full SDK state per frame as JSON, per-frame durability, easy to inspect with shell tools. That format is great for collection and debugging but is *not* what training pipelines expect.

LeRobot uses parquet files, normalized stats, image tensors, and explicit episode boundaries baked into the dataset. Converting once → training many times is the right trade-off.

## Inputs

- `ALLLOG/log<NNNN>.csv` — flat index across all episodes
- `ALLLOG/log<NNNN>/ep<NNN>/` — per-episode directories: `color/`, `depth/`, `state/`, optional `wrist/`

## Outputs

A LeRobot dataset under `$HF_LEROBOT_HOME` (default `~/.cache/huggingface/lerobot/`) with:

| Feature | Shape | Meaning |
|---|---|---|
| `observation.images.base` | (480, 640, 3) uint8 | base camera RGB |
| `observation.images.wrist` | (480, 640, 3) uint8 | wrist camera RGB *(only if present)* |
| `observation.state` | (7,) float32 | `[jp0..jp5, claw_amplitude]` |
| `action` | (7,) float32 | `[tgt_jp0..tgt_jp5, next_claw_amplitude]` |
| `task` | str | the language instruction you typed at recording time |


## Action representation: what we're predicting and why

The CSV gives us several candidates for the action signal. Default in this notebook:

**`tgt_jp0..tgt_jp5`** (target joint positions in radians) for the arm, with **gripper `claw_amplitude` from the next frame** as the 7th dimension.

Reasoning:

- **Joint-space, not Cartesian.** Joints map directly to motor commands, no IK ambiguity, no quaternion gymnastics. The arm in `grasp_to_the_bowl.py` uses `lebai.movej(joints_list, …)` exactly this way.
- **Target, not actual.** `tgt_jp*` is what the controller was *asked* to do; `jp*` is what the arm *actually did*. Imitation learning wants the command, not the response. (If `tgt_jp*` is empty for some firmwares during teaching mode, we fall back to `jp*` from the *next* frame — which is approximately what the command must have been.)
- **Next-frame gripper.** The gripper is a slow actuator. The command at time `t` shows up as the actual gripper state around `t+1`, so `next_row.claw_amplitude` is a clean approximation of "what the operator asked for at time `t`".

If you want to switch to delta joints or TCP poses later, this notebook is the only place to change it.


## 1. Setup


In [ ]:
# !pip install lerobot pandas opencv-python


In [ ]:
import json
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, HF_LEROBOT_HOME


## 2. Configuration

Edit the constants below.

- `LOG_ROOT`: where your `ALLLOG/` directory lives.
- `RUN_NUMBER`: which `log<NNNN>` to convert. Set to `None` to convert *all* runs into one dataset.
- `REPO_ID`: the dataset's name. The same string will be used as a HuggingFace repo id if you later push it.
- `INCLUDE_WRIST`: whether to include the wrist camera as a second image. Auto-detects if data is present.
- `INCLUDE_GRIPPER`: include `claw_amplitude` in state/action. Set False for gripper-free tasks.


In [ ]:
LOG_ROOT = Path("./Data")            # raw logs from save_state_img.py
RUN_NUMBER = None                    # int to select one log<NNNN>, None to merge all
REPO_ID = "local/lebai_duck_pick"    # dataset name; also used as HF repo id if pushed
INCLUDE_WRIST = True                 # auto-disabled if no wrist data is present
INCLUDE_GRIPPER = True
FPS = 10

# Image resolution from the camera service (matches save_state_img.py)
IMG_H, IMG_W = 480, 640

# Output location for LeRobot datasets
print(f"Datasets will be saved under: {HF_LEROBOT_HOME}")


## 3. Load the index CSV

The single CSV at `log<NNNN>.csv` is the source of truth — it indexes every frame across every episode. We parse it and check episode boundaries.


In [ ]:
def csv_for_run(run_number):
    return LOG_ROOT / f"log{run_number:04d}.csv"

if RUN_NUMBER is not None:
    csv_paths = [csv_for_run(RUN_NUMBER)]
else:
    csv_paths = sorted(LOG_ROOT.glob("log[0-9][0-9][0-9][0-9].csv"))
print(f"CSVs to convert: {[p.name for p in csv_paths]}")

dfs = []
for p in csv_paths:
    d = pd.read_csv(p)
    # The relative paths in the CSV are relative to log<NNNN>/, so we attach
    # the run directory so we can resolve them to absolute paths later.
    run_dir = LOG_ROOT / p.stem
    d["__run_dir__"] = str(run_dir)
    dfs.append(d)
df = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")


In [ ]:
# Episode summary — sanity check before we touch any images.
group = df.groupby(["__run_dir__", "episode"])
summary = group.agg(
    n_frames=("frame", "count"),
    task=("task", "first"),
    has_tgt_jp=("tgt_jp0", lambda s: s.notna().any()),
    has_wrist=("wrist", lambda s: s.notna().any() and (s.astype(str).str.len() > 0).any()),
).reset_index()
print(summary.to_string(index=False))


**What you're checking in that table**

- **`n_frames`**: episodes that are very short (a handful of frames) are usually accidental "I pressed SPACE twice by mistake" recordings. Drop them later, or just leave them — a few junk frames out of thousands is harmless.
- **`task`**: every episode should have a non-empty task string. If anything is empty, your converter will still run but training will be confused.
- **`has_tgt_jp`**: if this is `False` for some episodes, your firmware didn't populate `target_joint_pose` during teaching. The fallback path will use next-frame `jp*`, which is fine but worth knowing.
- **`has_wrist`**: confirms whether the wrist camera was present during that episode.


## 4. State and action vector builders


In [ ]:
def build_state(row):
    """observation.state: 6 joint positions + (optional) gripper amplitude."""
    s = [float(row[f"jp{i}"]) for i in range(6)]
    if INCLUDE_GRIPPER:
        amp = row.get("claw_amplitude", 0.0)
        s.append(float(amp) if pd.notna(amp) else 0.0)
    return np.array(s, dtype=np.float32)


def build_action(row, next_row):
    """action: target joints (fall back to next-frame jp*) + next-frame gripper.

    The fallback handles firmwares that don't populate target_joint_pose in
    teaching mode. next_row['jp*'] is approximately what the controller was
    asked to do, since the arm reaches the commanded pose within ~1 tick.
    """
    tgt = [row.get(f"tgt_jp{i}") for i in range(6)]
    if any(pd.isna(v) for v in tgt):
        tgt = [float(next_row[f"jp{i}"]) for i in range(6)]
    else:
        tgt = [float(v) for v in tgt]
    if INCLUDE_GRIPPER:
        amp = next_row.get("claw_amplitude", row.get("claw_amplitude", 0.0))
        tgt.append(float(amp) if pd.notna(amp) else 0.0)
    return np.array(tgt, dtype=np.float32)


def load_rgb(run_dir, rel_path):
    full = Path(run_dir) / rel_path
    bgr = cv2.imread(str(full), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Could not read image: {full}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


## 5. Quick check on a single frame before bulk conversion

Eyeball one frame's image, state, and action. This catches the most common bugs (wrong color channel, wrong column names, sign-flipped joints) before you spend an hour converting thousands of frames.


In [ ]:
import matplotlib.pyplot as plt

# Pick the first frame of the first episode
first_run = sorted(df["__run_dir__"].unique())[0]
ep_df = df[(df["__run_dir__"] == first_run) & (df["episode"] == df["episode"].min())]
ep_df = ep_df.sort_values("frame").reset_index(drop=True)
row = ep_df.iloc[0]
next_row = ep_df.iloc[1] if len(ep_df) > 1 else row

img = load_rgb(row["__run_dir__"], row["color"])
state = build_state(row)
action = build_action(row, next_row)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(img); ax.axis("off")
ax.set_title(f"task: {row['task']!r}")
plt.show()

print(f"state ({state.shape}): {np.round(state, 3)}")
print(f"action ({action.shape}): {np.round(action, 3)}")
print(f"state - action delta: {np.round(state - action, 4)}")


**The state-action delta is small but non-zero.** It should be:

- close to zero on each joint (the next-tick target is near the current position at 10 Hz)
- but not *exactly* zero (otherwise the policy is being asked to predict the input — a no-op task)

If it's exactly zero across many frames, your `tgt_jp*` columns are just copying `jp*` (some firmwares do that), and you should switch the action to `next_row[jp*]` explicitly via `INCLUDE_GRIPPER` settings or by editing `build_action`.


## 6. Build the LeRobot dataset


In [ ]:
state_dim = 7 if INCLUDE_GRIPPER else 6
action_dim = state_dim
has_wrist_data = INCLUDE_WRIST and df["wrist"].astype(str).str.len().gt(0).any()

features = {
    "observation.images.base": {
        "dtype": "image",
        "shape": (IMG_H, IMG_W, 3),
        "names": ["height", "width", "channel"],
    },
    "observation.state": {
        "dtype": "float32",
        "shape": (state_dim,),
        "names": ["state"],
    },
    "action": {
        "dtype": "float32",
        "shape": (action_dim,),
        "names": ["action"],
    },
}
if has_wrist_data:
    features["observation.images.wrist"] = {
        "dtype": "image",
        "shape": (IMG_H, IMG_W, 3),
        "names": ["height", "width", "channel"],
    }

# Wipe any previous version of this dataset on disk so we don't append accidentally.
out_path = HF_LEROBOT_HOME / REPO_ID
if out_path.exists():
    print(f"Removing existing dataset at {out_path}")
    shutil.rmtree(out_path)

dataset = LeRobotDataset.create(
    repo_id=REPO_ID,
    robot_type="lebai_lm3",
    fps=FPS,
    features=features,
    image_writer_threads=4,
    image_writer_processes=2,
)
print(f"Created empty dataset at {out_path}")
print(f"  state_dim={state_dim}  action_dim={action_dim}  wrist={has_wrist_data}")


## 7. Conversion loop

For each episode, walk frame by frame: load images, build vectors, call `add_frame`. After all frames of an episode are added, `save_episode()` writes the parquet shard and bumps the episode counter.

This step is the slow one (it decodes JPEGs and rewrites them as the dataset's preferred format). Expect ~50–200 frames/sec on a typical machine. A 10-episode dataset of 30 s episodes = 3000 frames = a few minutes.


In [ ]:
from tqdm import tqdm

total_frames = 0
total_episodes = 0

# Iterate (run, episode) so multi-run merging works
for (run_dir, ep_idx), ep_df in df.groupby(["__run_dir__", "episode"]):
    ep_df = ep_df.sort_values("frame").reset_index(drop=True)
    task = str(ep_df["task"].iloc[0])
    if not task or task == "nan":
        print(f"  skipping episode {ep_idx} in {Path(run_dir).name}: empty task")
        continue

    n = len(ep_df)
    if n < 5:
        print(f"  skipping episode {ep_idx} in {Path(run_dir).name}: only {n} frames")
        continue

    desc = f"{Path(run_dir).name}/ep{ep_idx:03d} [{task[:30]}]"
    for i in tqdm(range(n), desc=desc, leave=False):
        row = ep_df.iloc[i]
        next_row = ep_df.iloc[i + 1] if i + 1 < n else row

        frame = {
            "observation.images.base": load_rgb(row["__run_dir__"], row["color"]),
            "observation.state": build_state(row),
            "action": build_action(row, next_row),
            "task": task,
        }
        if has_wrist_data and isinstance(row["wrist"], str) and row["wrist"]:
            frame["observation.images.wrist"] = load_rgb(row["__run_dir__"], row["wrist"])

        dataset.add_frame(frame)

    dataset.save_episode()
    total_frames += n
    total_episodes += 1
    print(f"  saved {desc}: {n} frames")

print(f"\nDone. {total_episodes} episodes, {total_frames} frames total.")


## 8. Verify the dataset


In [ ]:
ds = LeRobotDataset(REPO_ID)
print(f"Episodes: {ds.num_episodes}")
print(f"Frames:   {ds.num_frames}")
print(f"FPS:      {ds.fps}")
print(f"Features: {list(ds.features.keys())}")

# Plot the first frame
sample = ds[0]
img = sample["observation.images.base"]
img_np = img.permute(1, 2, 0).cpu().numpy() if hasattr(img, "permute") else img
plt.figure(figsize=(6, 4))
plt.imshow(img_np)
plt.title(f"task: {sample['task']!r}")
plt.axis("off"); plt.show()

print(f"state:  {sample['observation.state'].numpy().round(3)}")
print(f"action: {sample['action'].numpy().round(3)}")


## 9. (Optional) Push to the HuggingFace Hub

Skip this cell if you're keeping the dataset local. Pushing to the Hub is useful when:

- Multiple people on your team want to train on the same data
- You want to fine-tune openpi later (the openpi configs reference HF repo IDs)
- You want the dataset to be a citable reference for a paper

Make a private repo if your data is sensitive (it usually is for medical/CCTV applications).


In [ ]:
# from huggingface_hub import login
# login()  # paste your token

# ds.push_to_hub(
#     tags=["lebai", "manipulation"],
#     private=True,
#     push_videos=True,
# )


## Next: train

Open `act_training.ipynb`, set `DATASET_REPO_ID` to the same `REPO_ID` you used here, and run.
